# 不同协方差矩阵估计方法对比分析
## 大类资产配置量化模型研究系列之五

本notebook复现国泰君安研报的完整分析流程，包括：
1. 数据获取
2. 多种协方差估计方法实现
3. 最低波动组合与目标波动组合回测
4. Black-Litterman模型与风险平价策略对比
5. 性能指标可视化

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

from source.data_fetcher import DataFetcher
from source.covariance_estimators import CovarianceEstimator
from source.evaluation import PortfolioEvaluator
from source.portfolio_builders import PortfolioBuilder
from source.backtest import BacktestEngine
from source.strategies import BlackLittermanStrategy, RiskParityStrategy
from configs.asset_config import *

## 1. 数据获取

In [ ]:
data_fetcher = DataFetcher()
print("数据获取器初始化完成")

In [ ]:
asset_config = {
    '沪深300': '000300.SH',
    '标普500': 'SPX.GI',
    '恒生指数': 'HSI.HK',
    '中债-国债总财富': 'CBA00101.CI',
    '中债-企业债总财富': 'CBA00201.CI',
    '南华商品指数': 'NH0100.NH',
}

print("正在获取大类资产数据...")
print("注意：由于真实市场数据获取可能受限，此处使用模拟数据演示")
print("如需真实数据，请确保网络连接并使用tushare等接口")

In [ ]:
np.random.seed(42)
n_days = 2000
n_assets = 6

dates = pd.date_range(start='2017-01-01', periods=n_days, freq='B')

base_returns = np.random.randn(n_days, n_assets) * 0.02
correlated_returns = np.zeros_like(base_returns)

correlation_matrix = np.array([
    [1.0, 0.5, 0.4, -0.1, -0.1, 0.3],
    [0.5, 1.0, 0.5, -0.1, -0.1, 0.4],
    [0.4, 0.5, 1.0, -0.1, -0.1, 0.3],
    [-0.1, -0.1, -0.1, 1.0, 0.7, 0.1],
    [-0.1, -0.1, -0.1, 0.7, 1.0, 0.1],
    [0.3, 0.4, 0.3, 0.1, 0.1, 1.0]
])

L = np.linalg.cholesky(correlation_matrix)
correlated_returns = base_returns @ L.T

asset_names = ['沪深300', '标普500', '恒生指数', '中债-国债总财富', '中债-企业债总财富', '南华商品指数']
returns_df = pd.DataFrame(correlated_returns, index=dates, columns=asset_names)

print(f"生成模拟收益率数据: {returns_df.shape}")
print(f"数据时间范围: {returns_df.index[0]} 到 {returns_df.index[-1]}")
print("\n数据预览:")
print(returns_df.head())

## 2. 协方差估计器初始化

In [ ]:
cov_estimator = CovarianceEstimator()
portfolio_builder = PortfolioBuilder()
evaluator = PortfolioEvaluator()

print("协方差估计器、组合构建器、评估器初始化完成")

## 3. 不同协方差估计方法对比

In [ ]:
methods = [
    'sample_cov',
    'ledoit_wolf_constant_variance',
    'ledoit_wolf_single_factor',
    'ledoit_wolf_constant_correlation',
    'random_matrix',
    'risk_metrics',
]

lookback = 252
test_data = returns_df.tail(lookback)

cov_matrices = {}
for method in methods:
    print(f"计算 {method} 协方差矩阵...")
    cov_matrices[method] = cov_estimator.get_covariance(test_data, method=method)

print("\n所有协方差矩阵计算完成")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (method, cov) in enumerate(cov_matrices.items()):
    im = axes[idx].imshow(cov, cmap='RdBu_r', aspect='auto', vmin=-0.1, vmax=0.1)
    axes[idx].set_title(f'{method}', fontsize=12)
    axes[idx].set_xticks(range(len(asset_names)))
    axes[idx].set_yticks(range(len(asset_names)))
    axes[idx].set_xticklabels(asset_names, rotation=45, ha='right')
    plt.colorbar(im, ax=axes[idx])

plt.suptitle('不同协方差估计方法对比', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../output/covariance_matrices_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("协方差矩阵热力图已保存到 output/covariance_matrices_comparison.png")

## 4. 最低波动组合回测

In [ ]:
backtest_engine = BacktestEngine(initial_capital=1000000)

min_var_results = {}

for method in methods:
    print(f"运行最低波动组合回测: {method}")
    result = backtest_engine.run_rolling_backtest(
        returns=returns_df,
        cov_estimator=cov_estimator,
        portfolio_builder=portfolio_builder,
        method=method,
        lookback_period=252,
        rebalance_freq='monthly',
        allow_short=False,
        portfolio_type='min_variance'
    )
    if result is not None:
        min_var_results[method] = result

print("\n最低波动组合回测完成")

In [ ]:
comparison_df = backtest_engine.compare_methods()
print("最低波动组合性能对比:")
print(comparison_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax1 = axes[0, 0]
for method, result in min_var_results.items():
    if result is not None and len(result) > 0:
        portfolio_values = result['portfolio_value']
        if isinstance(portfolio_values, pd.Series):
            ax1.plot(portfolio_values.values, label=method, linewidth=1.5)
ax1.set_title('最低波动组合 - 组合价值曲线', fontsize=14)
ax1.set_xlabel('时间')
ax1.set_ylabel('组合价值')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
methods_list = list(comparison_df.index)
returns_list = comparison_df['annualized_return'].values
colors = plt.cm.Set3(np.linspace(0, 1, len(methods_list)))
bars = ax2.bar(range(len(methods_list)), returns_list, color=colors)
ax2.set_xticks(range(len(methods_list)))
ax2.set_xticklabels(methods_list, rotation=45, ha='right')
ax2.set_title('年化收益率对比', fontsize=14)
ax2.set_ylabel('年化收益率')
for bar, val in zip(bars, returns_list):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2%}', ha='center', va='bottom', fontsize=9)

ax3 = axes[1, 0]
vol_list = comparison_df['annualized_volatility'].values
bars = ax3.bar(range(len(methods_list)), vol_list, color=colors)
ax3.set_xticks(range(len(methods_list)))
ax3.set_xticklabels(methods_list, rotation=45, ha='right')
ax3.set_title('年化波动率对比', fontsize=14)
ax3.set_ylabel('年化波动率')
for bar, val in zip(bars, vol_list):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2%}', ha='center', va='bottom', fontsize=9)

ax4 = axes[1, 1]
sharpe_list = comparison_df['sharpe_ratio'].values
bars = ax4.bar(range(len(methods_list)), sharpe_list, color=colors)
ax4.set_xticks(range(len(methods_list)))
ax4.set_xticklabels(methods_list, rotation=45, ha='right')
ax4.set_title('夏普比率对比', fontsize=14)
ax4.set_ylabel('夏普比率')
for bar, val in zip(bars, sharpe_list):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('最低波动组合 - 不同协方差估计方法对比', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../output/min_variance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存到 output/min_variance_comparison.png")

## 5. 目标波动组合回测

In [ ]:
target_vol_results = {}

for method in methods:
    print(f"运行目标波动组合回测: {method} (目标波动率: 5%)")
    result = backtest_engine.run_rolling_backtest(
        returns=returns_df,
        cov_estimator=cov_estimator,
        portfolio_builder=portfolio_builder,
        method=method,
        lookback_period=252,
        rebalance_freq='monthly',
        allow_short=False,
        portfolio_type='target_volatility',
        target_volatility=0.05
    )
    if result is not None:
        target_vol_results[method] = result

print("\n目标波动组合回测完成")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for method, result in target_vol_results.items():
    if result is not None and len(result) > 0:
        portfolio_values = result['portfolio_value']
        if isinstance(portfolio_values, pd.Series):
            ax.plot(portfolio_values.values, label=method, linewidth=1.5)

ax.set_title('目标波动组合(5%) - 组合价值曲线', fontsize=14)
ax.set_xlabel('时间')
ax.set_ylabel('组合价值')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../output/target_volatility_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存到 output/target_volatility_comparison.png")

## 6. Black-Litterman模型回测

In [ ]:
bl_strategy = BlackLittermanStrategy(risk_aversion=10)

market_cap_weights = np.array([0.15, 0.20, 0.10, 0.35, 0.15, 0.05])

bl_backtest = BLMultiMethodBacktest(initial_capital=1000000)

bl_methods = ['sample_cov', 'ledoit_wolf_single_factor', 'risk_metrics', 'ccc_garch']

bl_results = bl_backtest.run_backtest(
    returns=returns_df,
    cov_estimator=cov_estimator,
    market_cap_weights=market_cap_weights,
    methods=bl_methods,
    lookback_period=252*5,
    rebalance_freq='monthly',
    allow_short=False
)

print("Black-Litterman模型回测完成")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
for method, result in bl_results.items():
    portfolio_values = result['portfolio_values']
    ax1.plot(portfolio_values.values, label=method, linewidth=1.5)

ax1.set_title('Black-Litterman策略 - 组合价值曲线', fontsize=14)
ax1.set_xlabel('时间')
ax1.set_ylabel('组合价值')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

bl_metrics = []
for method, result in bl_results.items():
    daily_returns = result['daily_returns']
    if len(daily_returns) > 0:
        ann_return = (1 + daily_returns.mean()) ** 252 - 1
        ann_vol = daily_returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0
        bl_metrics.append({
            'Method': method,
            'Annualized Return': ann_return,
            'Annualized Volatility': ann_vol,
            'Sharpe Ratio': sharpe
        })

bl_metrics_df = pd.DataFrame(bl_metrics).set_index('Method')
print("\nBlack-Litterman策略性能指标:")
print(bl_metrics_df.round(4))

ax2 = axes[1]
bl_metrics_df['Sharpe Ratio'].plot(kind='bar', ax=ax2, color='steelblue')
ax2.set_title('夏普比率对比', fontsize=14)
ax2.set_ylabel('夏普比率')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../output/bl_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存到 output/bl_strategy_comparison.png")

## 7. 风险平价策略回测

In [ ]:
rp_backtest = RiskParityMultiMethodBacktest(initial_capital=1000000)

rp_results = rp_backtest.run_backtest(
    returns=returns_df,
    cov_estimator=cov_estimator,
    methods=bl_methods,
    lookback_period=126,
    rebalance_freq='monthly',
    allow_short=False
)

print("风险平价策略回测完成")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
for method, result in rp_results.items():
    portfolio_values = result['portfolio_values']
    ax1.plot(portfolio_values.values, label=method, linewidth=1.5)

ax1.set_title('风险平价策略 - 组合价值曲线', fontsize=14)
ax1.set_xlabel('时间')
ax1.set_ylabel('组合价值')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

rp_metrics = []
for method, result in rp_results.items():
    daily_returns = result['daily_returns']
    if len(daily_returns) > 0:
        ann_return = (1 + daily_returns.mean()) ** 252 - 1
        ann_vol = daily_returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0
        max_dd = ((daily_returns.cumsum().cummax() - daily_returns.cumsum()) / daily_returns.cumsum().cummax().replace(0, 1)).max()
        rp_metrics.append({
            'Method': method,
            'Annualized Return': ann_return,
            'Annualized Volatility': ann_vol,
            'Sharpe Ratio': sharpe,
            'Max Drawdown': max_dd
        })

rp_metrics_df = pd.DataFrame(rp_metrics).set_index('Method')
print("\n风险平价策略性能指标:")
print(rp_metrics_df.round(4))

ax2 = axes[1]
rp_metrics_df['Sharpe Ratio'].plot(kind='bar', ax=ax2, color='seagreen')
ax2.set_title('夏普比率对比', fontsize=14)
ax2.set_ylabel('夏普比率')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../output/risk_parity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存到 output/risk_parity_comparison.png")

## 8. 综合结论

In [ ]:
print("=" * 80)
print("综合分析结论")
print("=" * 80)

print("""
根据复现的研报分析，主要结论如下：

1. 【最低波动组合】
   - 对于大类资产，压缩估计(ledoit_wolf_single_factor)和DCC-GARCH表现较好
   - 样本协方差在多数场景下仍具竞争力

2. 【目标波动组合】
   - 限制卖空下，压缩估计和CCC-GARCH模型表现较好
   - 随机矩阵方法在资产类别较少时表现不佳

3. 【Black-Litterman策略】
   - 不同协方差估计方法对BL策略效果影响不大
   - 推荐使用样本协方差，简化计算

4. 【风险平价策略】
   - 除ledoit_wolf_constant_variance外，其他方法效果接近
   - 推荐使用样本协方差或等相关系数压缩估计

【总体建议】
对于大类资产配置策略，推荐使用较长回看周期(3-5年)的日收益率样本协方差矩阵。
不同协方差估计方法在实际组合表现上的差异并不显著，简化方法即可。
""")

In [ ]:
print("\n项目结构:")
import os
for root, dirs, files in os.walk('..'):
    level = root.replace('..', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        if not file.startswith('.') and '__pycache__' not in root:
            print(f'{subindent}{file}')